# Ligand Geometry Optimization

**Drug scaffold conformer relaxation with the uniqx SDK**

This notebook demonstrates a ligand geometry optimization workflow on the
4-anilinoquinazoline scaffold — the base for EGFR tyrosine kinase inhibitors
such as Gefitinib and Erlotinib.

## Pipeline

| Block | Algorithm |
|-------|-----------|
| 1. Spatial partitioning | Cell-linked neighbor lists |
| 2. Energy inference | GNN message passing (MACE-like) |
| 3. Geometry update | L-BFGS / FIRE |
| 4. Single-point validation | Restricted Hartree-Fock |

In [ ]:
import os

from uniqx import connect, submit, get, parse_buffer_view
from uniqx.domains.common.spatial import neighbor_list, pairwise_distances
from uniqx.domains.ml.gnn import GNNSpec, build_gnn_energy_module
from uniqx.domains.common.optimization import geometry_optimize
from uniqx.domains.chemistry.basis import extract_basis
from uniqx.domains.chemistry.hartree_fock import rhf_module

## 1. Define the molecular scaffold

The 4-anilinoquinazoline scaffold has 17 heavy atoms: a fused quinazoline
bicyclic ring system connected to an aniline ring via an amine bridge.

In [ ]:
positions = [
    [0.0, 0.0, 0.0],     # N1 (quinazoline)
    [1.3, 0.75, 0.0],    # C2
    [2.6, 0.0, 0.0],     # N3
    [2.6, -1.5, 0.0],    # C4
    [1.3, -2.25, 0.0],   # C4a
    [0.0, -1.5, 0.0],    # C8a
    [1.3, -3.75, 0.0],   # C5
    [0.0, -4.5, 0.0],    # C6
    [-1.3, -3.75, 0.0],  # C7
    [-1.3, -2.25, 0.0],  # C8
    [3.9, -2.25, 0.0],   # N (amine bridge)
    [5.2, -1.5, 0.0],    # C1'
    [6.5, -2.25, 0.0],   # C2'
    [7.8, -1.5, 0.0],    # C3'
    [7.8, 0.0, 0.0],     # C4'
    [6.5, 0.75, 0.0],    # C5'
    [5.2, 0.0, 0.0],     # C6'
]

atomic_numbers = [7, 6, 7, 6, 6, 6, 6, 6, 6, 6, 7, 6, 6, 6, 6, 6, 6]
n_atoms = len(positions)

print(f"Scaffold: 4-anilinoquinazoline")
print(f"Heavy atoms: {n_atoms}")
print(f"Elements: {set(atomic_numbers)} (C, N)")

## 2. Build neighbor list (CPU)

Spatial partitioning to identify interacting atom pairs within a cutoff distance.

In [ ]:
cutoff = 5.0  # Angstroms

edge_index, distances = neighbor_list(positions, cutoff)
n_edges = len(distances)

print(f"Cutoff: {cutoff} A")
print(f"Edges: {n_edges} ({n_edges / n_atoms:.1f} avg neighbors/atom)")
print(f"Min distance: {min(distances):.2f} A")
print(f"Max distance: {max(distances):.2f} A")

# Pairwise distance matrix for visualization
dist_matrix = pairwise_distances(positions)
print(f"\nDistance matrix ({n_atoms}x{n_atoms}):")
for row in dist_matrix[:5]:
    print("  ", " ".join(f"{d:5.2f}" for d in row[:5]), "...")

## 3. GNN energy inference (TPU)

Build a message passing GNN that predicts scalar energy from the molecular graph.
In production this would use pre-trained MACE or NequIP weights.

In [ ]:
spec = GNNSpec(
    node_feature_dim=16,
    edge_feature_dim=1,
    hidden_dim=32,
    output_dim=1,
    n_layers=3,
    activation="tanh",
)

edge_features = [[d] for d in distances]

gnn_mod, gnn_inputs, gnn_meta = build_gnn_energy_module(
    spec,
    n_nodes=n_atoms,
    n_edges=n_edges,
    edge_index=edge_index,
    edge_features=edge_features,
)

print(f"GNN architecture:")
print(f"  Layers: {spec.n_layers} message passing")
print(f"  Hidden dim: {spec.hidden_dim}")
print(f"  Activation: {spec.activation}")
print(f"  Module: {gnn_mod.name}")
print(f"  Runtime inputs: {len(gnn_inputs)}")

## 4. Geometry optimization (CPU, L-BFGS)

L-BFGS quasi-Newton optimization using the `optimize` primitive.

In [ ]:
opt_mod, opt_inputs, opt_meta = geometry_optimize(
    positions,
    atomic_numbers,
    method="lbfgs",
    max_steps=50,
    force_threshold=0.01,
)

print(f"Optimization:")
print(f"  Method: {opt_meta['method']}")
print(f"  Max steps: {opt_meta['max_steps']}")
print(f"  Force threshold: {opt_meta['force_threshold']} eV/A")
print(f"  DOF: {opt_meta['n_atoms'] * 3}")
print(f"  Module: {opt_mod.name}")

## 5. Hartree-Fock single-point validation

A short note: a tight-binding electronic-structure step (e.g. GFN2-xTB) is not
yet exposed in the SDK; using the available chemistry primitives instead.

Running RHF on the 17 heavy atom scaffold directly is open-shell (no
hydrogens), so we sanity-check the SDK round-trip on a small closed-shell
reference molecule (water) using `extract_basis` + `rhf_module`. For a real
drug-scaffold workflow, hydrogens would be added with an external tool such as
RDKit and a larger basis (cc-pVDZ) used.

In [ ]:
# Sanity-check RHF round-trip on a small closed-shell molecule.
rhf_atoms = [
    ("O", [0.0, 0.0, 0.0]),
    ("H", [0.757, 0.586, 0.0]),
    ("H", [-0.757, 0.586, 0.0]),
]

info = extract_basis(rhf_atoms, basis="sto-3g")
rhf_mod = rhf_module(rhf_atoms, info, max_iter=50)

def runtime_inputs_from(info):
    return [
        list(info.exps_flat),
        list(info.coeffs_flat),
        list(info.centers_flat),
        list(info.ang_flat),
        list(info.atom_coords_flat),
        list(info.charges_flat),
    ]

rhf_inputs = runtime_inputs_from(info)

print("Restricted Hartree-Fock (sanity check):")
print(f"  Molecule: H2O")
print(f"  Basis: sto-3g")
print(f"  Atoms: {info.n_atoms}")
print(f"  Basis functions: {info.n_basis}")
print(f"  Electrons: {info.n_electrons}")
print(f"  Nuclear repulsion: {info.nuclear_repulsion:.6f} a.u.")
print(f"  Module: {rhf_mod.name}")

## 6. Submit to uniqx

Trace each module, send the spec to the uniqx gateway, and collect results.
All electronic-structure and GNN evaluation runs server-side.

In [ ]:
from uniqx import login
endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)

# GNN energy
print("Submitting GNN energy inference...")
gnn_jid = submit(gnn_mod, gnn_inputs, client=client)
gnn_result = get(gnn_jid, client=client, timeout=300.0)
print(f"  Result: {gnn_result}")

# Geometry optimization
print("\nSubmitting L-BFGS optimization...")
opt_jid = submit(opt_mod, opt_inputs, client=client, iterations=50)
opt_result = get(opt_jid, client=client, timeout=300.0)
print(f"  Result: {opt_result}")

# RHF single point
print("\nSubmitting RHF single-point validation...")
rhf_jid = submit(rhf_mod, client=client, runtime_inputs=rhf_inputs)
rhf_result = get(rhf_jid, client=client, timeout=600.0)
payload = rhf_result.get("payload") or rhf_result.get("result_payload") or b""
if isinstance(payload, str):
    payload = payload.encode("utf-8", errors="replace")
rhf_energy = None
for ln in payload.decode("utf-8", errors="replace").splitlines():
    parsed = parse_buffer_view(ln)
    if parsed is not None and parsed[2]:
        rhf_energy = parsed[2][-1]
if rhf_energy is not None:
    print(f"  RHF energy: {rhf_energy:.6f} a.u.")
else:
    print(f"  Result: {rhf_result}")